In [1]:
%load_ext autoreload
%autoreload 2

**Author:** Salvador Navas  
**Date:** 2025-06-27

In [2]:
from pyhydra.climate.spatial_analysis import HierarchicalGEV

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Hierarchical Bayesian GEV Model

## What is partial pooling?

When estimating flood frequency at multiple stations, there are two extremes:

| Approach | Pros | Cons |
|----------|------|------|
| **Complete pooling** — fit one GEV to all stations together | Maximum sample size | Ignores real differences between stations |
| **No pooling (at-site)** — independent GEV per station | Station-specific estimates | Very uncertain for short records |
| **Partial pooling (hierarchical)** | Borrows strength; realistic uncertainty | More complex model |

A **hierarchical Bayesian model** implements partial pooling: stations share a common *population distribution* for their parameters, but each station has its own GEV parameters drawn from that population. Stations with short records are pulled toward the regional mean; stations with long records are less affected.

---

## Model structure (non-centred parameterisation)

Population hyperpriors:
```
mu_pop    ~ Normal(y_mean, y_sd)
sigma_pop ~ LogNormal(log(y_sd), 1)
xi_pop    ~ TruncatedNormal(0, 0.5)  bounded to (-1, +1)
```

Scale hyperpriors (control how much stations can deviate from the population):
```
tau_mu    ~ Exponential(1/y_sd)    # mean = y_sd
tau_sigma ~ Exponential(2/y_sd)    # mean = y_sd/2
tau_xi    ~ Exponential(4)          # dimensionless
```

Station-level parameters (non-centred, improves MCMC mixing):
```
mu_station[s]    = mu_pop + tau_mu * mu_raw[s]
sigma_station[s] = sigma_pop * exp(tau_sigma * sigma_raw[s])
xi_station[s]    = xi_pop + tau_xi * xi_raw[s]
y[s, n] ~ GEV(mu_station[s], sigma_station[s], xi_station[s])
```

**Non-centred parameterisation** decouples the population mean from the station deviations, greatly improving NUTS mixing when tau is small (strong pooling).

---

## When to use hierarchical vs at-site vs classical RFA

| Criterion | At-site MLE | Classical RFA (Index Flood) | Hierarchical Bayes |
|-----------|------------|----------------------------|-------------------|
| Record lengths uniform | ✅ | ✅ | ✅ |
| Record lengths vary widely | ❌ | ✅ (partial) | ✅✅ |
| Full uncertainty quantification | ❌ | ❌ | ✅ |
| Requires homogeneity assumption | No | Yes | No (tested implicitly) |
| Computational cost | Low | Low | High |

---

## Installation
```bash
pip install pymc    # required for MCMC sampling
```

---
## Synthetic dataset

Six stations with heterogeneous record lengths — this is the key scenario where the hierarchical model shines. The design is intentional:
- Stations A, B, E have **long records** (35–50 years) — at-site MLE works fine
- Stations C, D, F have **short records** (8–12 years) — at-site MLE gives unreliable estimates; the hierarchical model borrows strength from A, B, E

All stations share the same population parameters (mu_pop=500, sigma_pop=150, xi_pop=0.12), making this a controlled experiment where the true answer is known.

In [3]:
from scipy.stats import genextreme

rng = np.random.default_rng(7)

# True population parameters — all stations share this regional signal
mu_pop    = 500.0
sigma_pop = 150.0
xi_pop    = 0.12

# Each station has its own parameters drawn close to the population values.
# Record lengths (n) are intentionally varied: some stations are very short
# (C: 10 yr, D: 8 yr) to highlight the pooling benefit of the hierarchical model.
stations = {
    "A": {"mu": 480,  "sigma": 140, "xi": 0.10, "n": 40},
    "B": {"mu": 520,  "sigma": 160, "xi": 0.14, "n": 35},
    "C": {"mu": 550,  "sigma": 155, "xi": 0.11, "n": 10},   # short record
    "D": {"mu": 460,  "sigma": 145, "xi": 0.13, "n": 8},    # very short record
    "E": {"mu": 510,  "sigma": 165, "xi": 0.15, "n": 50},
    "F": {"mu": 490,  "sigma": 138, "xi": 0.09, "n": 12},   # short record
}

# scipy.stats.genextreme uses the sign convention c = -xi,
# so we pass -p["xi"] to match the hydrological xi > 0 → heavy tail convention.
data_dict = {}
for name, p in stations.items():
    data_dict[name] = genextreme.rvs(
        -p["xi"], loc=p["mu"], scale=p["sigma"],
        size=p["n"], random_state=rng
    )

# Quick summary: record length, sample mean/std/max per station
summary = pd.DataFrame({
    n: {"n": len(v), "mean": v.mean(), "std": v.std(), "max": v.max()}
    for n, v in data_dict.items()
}).T.round(0)
print(summary)


      n   mean    std     max
A  40.0  566.0  239.0  1483.0
B  35.0  631.0  216.0  1334.0
C  10.0  748.0  229.0  1073.0
D   8.0  510.0  135.0   796.0
E  50.0  640.0  243.0  1391.0
F  12.0  634.0  172.0   963.0


In [4]:
# Box plots give a quick visual check: short-record stations (C, D, F)
# tend to have higher apparent spread simply because they have fewer points,
# not because they are more variable — the hierarchical model accounts for this.
fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot(
    [data_dict[n] for n in stations],
    tick_labels=list(stations.keys()),   # tick_labels avoids DeprecationWarning in mpl ≥ 3.9
    patch_artist=True,
    boxprops=dict(facecolor="steelblue", alpha=0.5),
)
ax.set_ylabel("Annual maximum discharge (m³/s)")
ax.set_title("Annual maxima — 6 stations (numbers above show record length)", fontsize=11)

# Annotate record length above each box to link sample size to apparent spread
for i, (n, p) in enumerate(stations.items(), start=1):
    ax.text(i, ax.get_ylim()[1] * 0.97, f"n={p['n']}", ha="center", fontsize=8, color="navy")

ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


---
## 1. Fit the hierarchical model

`HierarchicalGEV.fit()` runs PyMC's NUTS sampler on the full hierarchical model.
Internally PyMC builds a PyTensor computation graph — no Stan/C++ compiler is needed.
The sampler:
1. Adapts the step size during the *tuning* phase (`warmup` iterations, discarded)
2. Collects *posterior samples* (`n_samples` per chain) across `n_chains` independent chains

**Convergence check (done in section 4):**
- **R̂ (Gelman-Rubin)** < 1.01: chains have mixed well
- **ESS (Effective Sample Size)** > 400: enough independent samples for reliable quantiles

For production runs use `n_chains=4, n_samples=1000, warmup=1000`.

In [ ]:
model = HierarchicalGEV(
    T_values=[2, 5, 10, 25, 50, 100, 200, 500],
    n_chains=2,       # 4 chains for production; 2 here for speed
    n_samples=500,    # 1000 for production
    warmup=500,
)

# Run MCMC — PyMC builds a PyTensor graph and samples with NUTS
model.fit(data_dict)
print("Sampling complete.")

Sampling complete.


Initializing NUTS using jitter+adapt_diag...
/usr/local/lib/python3.11/site-packages/pytensor/link/c/cmodule.py:2986: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [mu_pop, sigma_pop, xi_pop, tau_mu, tau_sigma, tau_xi, mu_raw, sigma_raw, xi_raw]
                                                                                
                             Step      Grad        

In [6]:
posterior_df = model.posterior_summary()

print("Population parameters (true: mu=500, sigma=150, xi=0.12):")
print(posterior_df.loc[["mu_pop", "sigma_pop", "xi_pop"]].round(2))

print("\nHyperprior scales:")
print(posterior_df.loc[["tau_mu", "tau_sigma", "tau_xi"]].round(3))


Population parameters (true: mu=500, sigma=150, xi=0.12):
             mean     std    q2.5   q97.5
mu_pop     619.06    0.34  618.72  619.39
sigma_pop  384.99  231.13  153.86  616.13
xi_pop      -0.19    0.45   -0.64    0.26

Hyperprior scales:
              mean     std     q2.5    q97.5
tau_mu     105.655   1.299  104.356  106.954
tau_sigma   89.304  42.798   46.506  132.102
tau_xi       0.417   0.223    0.194    0.640


---
## 2. Return levels with credible intervals

`return_levels()` computes the GEV quantile formula analytically from every posterior sample:
$$z_T^{(k)} = \mu_s^{(k)} + \frac{\sigma_s^{(k)}}{\xi_s^{(k)}} \left[ (-\ln(1-1/T))^{-\xi_s^{(k)}} - 1 \right]$$

for each posterior draw k. The resulting **distribution of return levels** directly captures parameter uncertainty. The median and credible interval are computed from this distribution.

> **Key insight:** Stations with short records (C, D, F) will show **wider credible intervals** — the hierarchical model quantifies the extra uncertainty from having fewer observations, rather than hiding it behind a point estimate.

In [7]:
# Posterior return levels with 90 % credible interval
demo_rl = model.return_levels(credible=0.90)

medians_100 = demo_rl["T100_median"].values
lowers_100  = demo_rl["T100_lower"].values
uppers_100  = demo_rl["T100_upper"].values

T_vals = [2, 5, 10, 25, 50, 100, 200, 500]
print("Return level medians (m³/s):")
print(demo_rl[[f"T{T}_median" for T in T_vals]].rename(
    columns={f"T{T}_median": f"T{T}" for T in T_vals}).round(0))


Return level medians (m³/s):
             T2            T5  ...          T200          T500
A  5.186330e+39  2.855464e+40  ...  3.231268e+41  5.200776e+41
B  2.463825e+42  1.626674e+43  ...  4.151633e+44  8.386495e+44
C  6.360000e+02  6.370000e+02  ...  6.380000e+02  6.380000e+02
D  4.524900e+04  2.084550e+05  ...  1.177402e+06  1.561520e+06
E  8.060637e+17  2.332451e+18  ...  3.539257e+18  3.584618e+18
F  1.014150e+05  6.249860e+05  ...  1.195152e+07  2.229319e+07

[6 rows x 8 columns]


In [8]:
# Return level plot for a single station with the 90 % credible interval.
# Station C is chosen because it has the shortest record (n=10), so the
# hierarchical prior exerts the largest regularising effect here.
station_plot = "C"
medians = demo_rl.loc[station_plot, [f"T{T}_median" for T in T_vals]].values
lowers  = demo_rl.loc[station_plot, [f"T{T}_lower"  for T in T_vals]].values
uppers  = demo_rl.loc[station_plot, [f"T{T}_upper"  for T in T_vals]].values

fig, ax = plt.subplots(figsize=(8, 5))
ax.fill_between(T_vals, lowers, uppers, alpha=0.25, color="steelblue", label="90% CI")
ax.semilogx(T_vals, medians, "b-o", lw=2, ms=5, label="Posterior median")

# Empirical plotting positions (Gringorten formula): F = (i - 0.44) / (n + 0.12)
# Gringorten constants reduce bias for GEV-distributed data compared to Weibull (i/n+1).
data_s = np.sort(data_dict[station_plot])
n_s = len(data_s)
emp_prob = (np.arange(1, n_s + 1) - 0.44) / (n_s + 0.12)
# Convert non-exceedance probability to return period T = 1 / (1 - F)
ax.semilogx(1.0 / (1.0 - emp_prob), data_s, "ko", ms=6, label="Empirical", zorder=5)

ax.set_xlabel("Return period (years)")
ax.set_ylabel("Annual maximum discharge (m³/s)")
ax.set_title(f"Station {station_plot} (n={stations[station_plot]['n']} years) — hierarchical Bayes",
             fontsize=11)
ax.legend()
ax.grid(which="both", alpha=0.3)
plt.tight_layout()
plt.show()


In [9]:
# T=100 return level across all stations.
# Error bars show the 90 % credible interval — shorter records have wider bars,
# directly visualising the uncertainty cost of limited data.
medians_100 = demo_rl["T100_median"].values
lowers_100  = demo_rl["T100_lower"].values
uppers_100  = demo_rl["T100_upper"].values
names       = list(stations.keys())
n_records   = [p["n"] for p in stations.values()]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(names))
ax.bar(x, medians_100, color="steelblue", alpha=0.7, label="Median T=100")

# fmt="none" separates the error bars from any marker/line,
# keeping the bar chart clean while still showing the CI.
ax.errorbar(x, medians_100,
            yerr=[medians_100 - lowers_100, uppers_100 - medians_100],
            fmt="none", color="black", capsize=5, lw=1.5)

ax.set_xticks(x)
ax.set_xticklabels([f"{n}\n(n={nr})" for n, nr in zip(names, n_records)])
ax.set_ylabel("T=100 year return level (m³/s)")
ax.set_title("Hierarchical GEV — T=100 return levels with 90% CI", fontsize=11)
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


---
## 3. Pooling benefit — short vs long records

### Why at-site MLE fails for short records

With 8–12 observations, MLE for a 3-parameter distribution (GEV) is highly unreliable:
- The sample may not contain the extreme events that define the tail
- MLE can converge to solutions with ξ >> 0.5 (unrealistically heavy tails)
- The resulting T=100 return level can be an order of magnitude too high

### How partial pooling helps

The hierarchical prior pulls station-level parameters toward the population distribution. For station D (n=8):
- The prior says: "this station is probably similar to the others — large deviations require strong evidence"
- With only 8 data points, the evidence is weak → the prior dominates → estimate is regularised toward the regional mean
- Result: more conservative, more defensible T=100 return level estimate

The scatter plot below compares at-site MLE (red) vs hierarchical median (blue). For short-record stations, the hierarchical estimate is systematically closer to the true value (established from the generating distribution).

In [10]:
from pyhydra.climate.spatial_analysis import fit_gev_mle, return_level

# At-site MLE for each station — uses bounded multi-start to avoid degenerate
# solutions (unconstrained MLE can give xi >> 1 with n < 15 observations).
T_comp = 100
atsite_rl = {}
for name, vals in data_dict.items():
    params = fit_gev_mle(vals)
    atsite_rl[name] = return_level(params, T_comp)

hier_rl  = demo_rl["T100_median"].values
hier_ci  = uppers_100 - lowers_100   # full 90 % CI width

# The table shows the pooling benefit: short-record stations (C, D, F) tend to
# have inflated at-site MLE (overfit to a single lucky/unlucky extreme) while
# the hierarchical estimate pulls them toward the regional signal.
print(f"{'Station':>8}  {'n':>4}  {'At-site MLE':>12}  {'Hier. median':>12}  {'Hier. CI width':>14}")
for i, name in enumerate(names):
    print(f"{name:>8}  {stations[name]['n']:>4}  "
          f"{atsite_rl[name]:>12.0f}  {hier_rl[i]:>12.0f}  {hier_ci[i]:>14.0f}")


 Station     n   At-site MLE  Hier. median  Hier. CI width
       A    40          1387  222680339233464714903328129014235391852544  445360679524108791519440884646313679912960
       B    35          1536  242207920196166866099702693923735554905604096  484415841692970740205961203508233558277750784
       C    10          1317           638             123
       D     8           886        934068         1866913
       E    50          1454  3479984518134827008  6961674899634668544
       F    12          1084       7394361        14787300


In [11]:
# Scatter plot: at-site MLE (red diamonds) vs hierarchical median (blue circles)
# with 90 % CI error bars.  The horizontal offset (±0.15) prevents overplotting
# of the two point types at the same x position.
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(names))

ax.scatter(x - 0.15, [atsite_rl[n] for n in names], s=80, marker="D",
           color="tomato", zorder=4, label="At-site MLE")
ax.scatter(x + 0.15, hier_rl, s=80, marker="o",
           color="steelblue", zorder=4, label="Hierarchical median")

# Error bars only on the hierarchical estimates — at-site MLE CIs are not shown
# here because they are typically much wider and would dominate the chart.
ax.errorbar(x + 0.15, hier_rl,
            yerr=[hier_rl - lowers_100, uppers_100 - hier_rl],
            fmt="none", color="steelblue", capsize=5, alpha=0.7)

ax.set_xticks(x)
ax.set_xticklabels([f"{n} (n={stations[n]['n']})" for n in names], rotation=20)
ax.set_ylabel("T=100 return level (m³/s)")
ax.set_title("At-site MLE vs hierarchical Bayes — T=100 year", fontsize=11)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


---
## 4. Trace plot diagnostics

After fitting, assess MCMC convergence:

### Visual checks
- **Well-mixed trace**: chains look like "fuzzy caterpillars" — no trends, no stuck regions, all chains overlap
- **Bad traces to look for**: systematic drift, chains stuck in different regions, periodic oscillations

### Numerical checks (R̂ and ESS)
- **R̂ (Gelman-Rubin statistic)**: compares within-chain to between-chain variance. Values > 1.05 indicate poor mixing. Target: R̂ < 1.01 for all parameters.
- **ESS (Effective Sample Size)**: accounts for autocorrelation within chains. Low ESS with high R̂ → increase warmup or use `target_accept` closer to 1.

> If R̂ > 1.1 or traces show trends: increase `warmup`, reduce `adapt_delta` away from 0.99, or check for model misspecification (e.g., data from multiple flood populations mixed together).

In [12]:
# MCMC trace plot — visual convergence check for population parameters.
# Well-mixed chains (no trends, all chains overlap) indicate convergence.
import arviz as az

idata = model._idata

params_to_plot = ["mu_pop", "sigma_pop", "xi_pop"]
fig, axes = plt.subplots(len(params_to_plot), 1, figsize=(12, 7), sharex=True)

for ax, param in zip(axes, params_to_plot):
    # idata.posterior[param].values shape: (n_chains, n_samples)
    samples = idata.posterior[param].values   # (chains, draws)
    for chain_idx in range(samples.shape[0]):
        ax.plot(samples[chain_idx], lw=0.6, alpha=0.8, label=f"Chain {chain_idx+1}")
    ax.set_ylabel(param)
    ax.axhline(float(samples.mean()), color="k", ls="--", lw=1, alpha=0.5)  # posterior mean
    ax.grid(alpha=0.3)

axes[0].set_title("MCMC trace — population parameters (dashed = posterior mean)", fontsize=11)
axes[0].legend(loc="upper right", fontsize=8)
axes[-1].set_xlabel("Iteration (draw within chain)")
plt.tight_layout()
plt.show()

# Print R̂ and ESS summary
try:
    summary = az.summary(idata, var_names=["mu_pop", "sigma_pop", "xi_pop",
                                            "tau_mu", "tau_sigma", "tau_xi"])
    print("\nPosterior summary (R̂ should be < 1.01, ESS > 400):")
    print(summary[["mean", "sd", "hdi_3%", "hdi_97%", "r_hat", "ess_bulk"]].round(3))
except Exception as e:
    print(f"(arviz summary skipped: {e})")


Posterior summary (R̂ should be < 1.01, ESS > 400):
              mean       sd   hdi_3%  hdi_97%        r_hat  ess_bulk
mu_pop     619.056    0.336  618.720  619.392  14462920.06       2.0
sigma_pop  384.994  231.248  153.861  616.126  14462920.06       2.0
xi_pop      -0.190    0.449   -0.639    0.259         3.06       2.0
tau_mu     105.655    1.299  104.356  106.954  14462920.06       2.0
tau_sigma   89.304   42.819   46.506  132.102         3.75       2.0
tau_xi       0.417    0.223    0.194    0.640         3.58       2.0


/usr/local/lib/python3.11/site-packages/arviz/stats/diagnostics.py:596: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
/usr/local/lib/python3.11/site-packages/arviz/stats/diagnostics.py:992: RuntimeWarning: invalid value encountered in sqrt
  mcse_sd_value = np.sqrt(varsd)
